#**Week 5 Day 5**

1. Nested CV: tune hyperparameters AND evaluate the tuned model correctly
2. GridSearchCV over the whole Pipeline: tune preprocessing AND model params together
3. Save pipeline to disk with joblib: load it, run predictions on new raw data



In [15]:
import numpy as np
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Roadmap/Datasets/titanic/train.csv")

df['Sex_encoded'] = df['Sex'].map({"male":0, "female":1})
features = ['Pclass', 'Age', 'Fare', 'Sex_encoded']

X = df[features]
y = df['Survived']

In [16]:
# Pipeline

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3, random_state= 42)

pipeline = Pipeline([
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(random_state= 42))
])

#Pipeline fit
pipeline.fit(X_train, y_train)


Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler()),
                ('model', RandomForestClassifier(random_state=42))])

In [17]:
#pipeline predict
y_pred = pipeline.predict(X_test)

print(f"Accuracy Score of Pipeline: {accuracy_score(y_test, y_pred):.3f}")


Accuracy Score of Pipeline: 0.802


#### Task 1  
1. Nested CV Outer loop evaluates model performance.
2. Inner loop tunes hyperparameters.
- Why do you need two loops instead of one?


1. Pipeline (imputer → scaler → model)
  -  ↓
2. GridSearchCV wraps Pipeline (inner loop: finds best params)
  -    ↓
3. cross_val_score wraps GridSearchCV (outer loop: evaluates performance)

In [18]:
from sklearn.model_selection import cross_val_score, KFold, GridSearchCV
# Step 1  define what to tune
param_grid = {
    'model__n_estimators':[100,200],
    'model__max_depth':[3,6]
}


#Step 2 - inner loop(tune hyperparameters)
inner_cv = KFold(n_splits= 3, shuffle= True, random_state=42)
grid_search = GridSearchCV(pipeline, param_grid, cv= inner_cv)


# STep 3- outer loop is used to evaluate
outer_cv = KFold(n_splits= 5,shuffle = True, random_state=42)
nested_scores = cross_val_score(grid_search, X,y , cv= outer_cv)

print(f"Nested CV score: {np.mean(nested_scores):.3f}")

Nested CV score: 0.829


####Task 2  
1. GridSearchCV over full Pipeline Tune both preprocessing params AND model params together in one GridSearch.


In [20]:
param_grid = {
    'imputer__strategy':['mean','median'],
    'model__n_estimators':[100,200],
    'model__max_depth':[3,6]
}
grid_search = GridSearchCV(pipeline,param_grid, cv=5)
grid_search.fit(X_train, y_train)


print(f"Best params: {grid_search.best_params_}")
print(f"Best score: {grid_search.best_score_:.3f}")

Best params: {'imputer__strategy': 'median', 'model__max_depth': 6, 'model__n_estimators': 100}
Best score: 0.831


#### Task 3
1. Save and load Pipeline Use joblib to save trained pipeline to disk, load it, predict on new raw data.


In [23]:
import joblib

#save
joblib.dump(grid_search.best_estimator_, 'titanic_pipeline.pk1')


#load
loaded_pipeline = joblib.load('titanic_pipeline.pk1')


#predicting on raw data
y_pred_loaded = loaded_pipeline.predict(X_test)
print(f"Loaded Pipeline accuracy: {accuracy_score(y_test, y_pred_loaded):.3f}")

Loaded Pipeline accuracy: 0.810


#### Task 4
## Nested CV, GridSearchCV over Pipeline, joblib
1. Nested CV separates tuning from evaluation. Outer loop gives unbiased
performance estimate:test fold never seen during tuning.
2. Inner loop finds best hyperparameters.
3. Without nested CV, GridSearch
- optimises params for one specific test set → optimistic bias → score drops on
new data.
### Nested CV
- When to use vs regular CV:
1. Use when you need a trustworthy performance estimate on a tuned model.
2. Regular CV + GridSearch on same test set = optimistically biased score.
3. Nested CV = honest score.
- itanic example: 0.802 (single split) vs
0.829 (nested CV), nested CV gave more reliable estimate.
### GridSearchCV over Pipeline
- Advantage of double underscore params:
1. Double underscore syntax (stepname__paramname) tunes preprocessing
AND model params together in one search.
- e.g. 'imputer__strategy' tunes imputation,
-  'model__max_depth' tunes
the model
- found median imputation + max_depth=6 → 0.831 best score.
### joblib
1. Always save full pipeline, never just model weights.
2. Pipeline stores exact imputer strategy + scaler's learned mean/std +
model weights together.
3. Loading pipeline on new raw data → one predict() call handles everything correctly.
4. Saving only model weights → must manually recreate preprocessing
in exact same order with exact same params → error prone in production.